In [ ]:
use role sysadmin;
use warehouse compute_wh;
use schema legacy_db.clean_sch;

****
### Dataset Validation from Stages
****

In [ ]:
select 
    t.$1,t.$2,t.$3,t.$4,t.$5,t.$6,t.$7,
    current_timestamp(),
    metadata$file_row_number,
    metadata$filename
from @legacy_db.source_sch.my_stage/customer/ as t;

In [ ]:
select 
    t.$1,t.$2,t.$3,t.$4,t.$5,t.$6,t.$7,t.$8,
    current_timestamp(),
    metadata$file_row_number,
    metadata$filename
from @legacy_db.source_sch.my_stage/order/ as t;

In [ ]:
select 'customer_stage' as source, count(*) as row_count from @legacy_db.source_sch.my_stage/customer/
union all
select 'order_stage', count(*) from @legacy_db.source_sch.my_stage/order/;

****
### Raw Layer Validation
****

In [ ]:
select 'customers_raw' as table_name, count(*) as row_count from legacy_db.raw_sch.customers_raw
union all
select 'customer_raw_stream', count(*) from legacy_db.raw_sch.customer_raw_stream
union all
select 'orders_raw', count(*) from legacy_db.raw_sch.orders_raw
union all
select 'orders_raw_stream', count(*) from legacy_db.raw_sch.orders_raw_stream;

****
### Clean Layer Validation
****

In [ ]:
select 'customers_clean' as table_name, count(*) as row_count from legacy_db.clean_sch.customers_clean
union all
select 'customers_clean_stream', count(*) from legacy_db.clean_sch.customers_clean_stream
union all
select 'orders_clean', count(*) from legacy_db.clean_sch.orders_clean
union all
select 'orders_clean_stream', count(*) from legacy_db.clean_sch.orders_clean_stream;

****
### Consumption Layer Validdation
****

In [ ]:
EXECUTE TASK LEGACY_DB.RAW_SCH.ROOT_TASK;

In [ ]:
select 'customer_dim' as table_name, count(*) as row_count from legacy_db.consumption_sch.customer_dim
union all
select 'date_dim', count(*) from legacy_db.consumption_sch.date_dim
union all
select 'priority_dim', count(*) from legacy_db.consumption_sch.priority_dim
union all
select 'order_fact', count(*) from legacy_db.consumption_sch.order_fact;

****
### Validate Task History
****

In [ ]:
SELECT name, state, scheduled_time, completed_time, error_code, error_message
FROM TABLE(LEGACY_DB.INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE database_name = 'LEGACY_DB'
ORDER BY scheduled_time DESC;

In [ ]:
SELECT file_name, stage_location, status, row_count, row_parsed, error_count, first_error_message, last_load_time
FROM TABLE(LEGACY_DB.INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'LEGACY_DB.RAW_SCH.CUSTOMERS_RAW',
    START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
UNION ALL
SELECT file_name, stage_location, status, row_count, row_parsed, error_count, first_error_message, last_load_time
FROM TABLE(LEGACY_DB.INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'LEGACY_DB.RAW_SCH.ORDERS_RAW',
    START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
ORDER BY last_load_time DESC;

****
### Validate Above Data
****
Now resume all the tasks from 05 notebook, then validate all the data from this notebook. Once done - then load more data into stage - customer & order folder & again validate all the data.

Once everything is completed, suspend all the tasks from the 05 notebook.